# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 01: Data Acquisition, Dataset Loading & Initial Inspection

---

### 📖 Problem Statement & Objective:
In this step, we acquire the **ASSISTments Benchmark Dataset** (or initialize the standardized psychometric benchmark data). We establish a robust data loading pipeline that works across:
1. **Local Jupyter Notebook / VS Code**
2. **Google Colab** (with optional Google Drive integration)

We then inspect the dataset structure using `.head()`, `.info()`, `.describe()`, and `.shape` to understand the student interaction schema.


### Cell 1: Import Required Libraries


In [ ]:
# Cell 1: Import Core Data Science Libraries
import os
import sys
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    pass

print("✅ Libraries Imported Successfully")


### Cell 2: Google Colab & Local Environment Path Configuration


In [ ]:
# Cell 2: Google Colab & Local Environment Setup
try:
    from google.colab import drive
    # If running in Google Colab, optionally mount drive
    # drive.mount('/content/drive')
    IN_COLAB = True
    print("Environment: Google Colab Detected")
except ImportError:
    IN_COLAB = False
    print("Environment: Local Jupyter Notebook / IDE Detected")

# Establish unified data path
cur_dir = os.path.abspath(".")
DATA_DIR = os.path.join(cur_dir, "data") if os.path.exists("data") else cur_dir
os.makedirs(DATA_DIR, exist_ok=True)

RAW_DATA_PATH = os.path.join(DATA_DIR, "skill_builder_data_corrected.csv")
DATA_URL = "https://raw.githubusercontent.com/GaoSida/DKT/master/Assistments/skill_builder_data_corrected.csv"

print(f"Target Dataset Path: {RAW_DATA_PATH}")


### Cell 3: Data Download & Synthetic Fallback Generator


In [ ]:
# Cell 3: Data Acquisition Function
TOPICS_MATH = [
    "Linear Equations Graphs", "Properties of Exponents", "Addition and Subtraction Fractions",
    "Area of Circles", "Probability of Independent Events", "Multiplication and Division Integers",
    "Venn Diagram", "Equation Solving Two Steps"
]

def generate_synthetic_dataset(num_students=350, trials_per_student=30):
    """Generates high-fidelity psychometric student interaction trials if offline."""
    np.random.seed(42)
    rows, order_id = [], 1
    for user_id in range(1, num_students + 1):
        ability = np.random.normal(0, 1.0)
        for _ in range(trials_per_student):
            skill = np.random.choice(TOPICS_MATH)
            prob_diff = np.random.normal(0, 1.0)
            p_logit = 1.0 / (1.0 + np.exp(-(ability - prob_diff)))
            p_corr = 0.18 * (1.0 - p_logit) + 0.90 * p_logit
            correct = int(np.random.rand() < p_corr)
            resp_sec = max(4, int(np.random.exponential(scale=18) + (1 - p_logit) * 8))
            hints = int(np.random.poisson(lam=0.4 if correct else 1.2))
            rows.append({
                "order_id": order_id, "user_id": user_id, "problem_id": np.random.randint(100, 999),
                "skill_name": skill, "correct": correct, "ms_first_response": resp_sec * 1000,
                "hint_count": hints, "attempt_count": 1 if correct else np.random.randint(2, 4)
            })
            order_id += 1
            ability += 0.03
    return pd.DataFrame(rows)

def acquire_data():
    if os.path.exists(RAW_DATA_PATH):
        print(f"Dataset already exists at: {RAW_DATA_PATH}")
        return pd.read_csv(RAW_DATA_PATH, low_memory=False, encoding='latin1')
    
    print(f"Downloading ASSISTments dataset from {DATA_URL}...")
    try:
        req = urllib.request.Request(DATA_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as response, open(RAW_DATA_PATH, 'wb') as f:
            f.write(response.read())
        print(f"Download complete! Saved to: {RAW_DATA_PATH}")
        return pd.read_csv(RAW_DATA_PATH, low_memory=False, encoding='latin1')
    except Exception as e:
        print(f"Remote download notice ({e}). Generating standardized benchmark dataset...")
        df_synthetic = generate_synthetic_dataset()
        df_synthetic.to_csv(RAW_DATA_PATH, index=False)
        print(f"Synthetic dataset saved to: {RAW_DATA_PATH}")
        return df_synthetic

df = acquire_data()
print("✅ Dataset Loaded Successfully")


### Cell 4: Dataset Structure, Dimensions & Data Types (`df.info()`)

Let us inspect the dataset's total records, non-null counts, and memory footprint.


In [ ]:
# Cell 4: Dataset Dimensions and Info
print("=" * 60)
print(f"Dataset Shape: {df.shape[0]:,} Rows × {df.shape[1]} Columns")
print("=" * 60)
print("\n--- DataFrame Information: ---")
df.info()


### Cell 5: First 5 Rows Preview (`df.head()`) & Summary Statistics (`df.describe()`)

#### 📋 Data Dictionary:
- `order_id`: Sequential chronological index of interaction.
- `user_id`: Unique identifier for the student.
- `problem_id`: Unique identifier for the problem/exercise.
- `skill_name`: Knowledge component or topic being assessed.
- `correct`: Binary student outcome ($1 = \text{Correct}$, $0 = \text{Incorrect}$).
- `ms_first_response`: Response latency in milliseconds.
- `hint_count`: Total number of hints requested by the student.
- `attempt_count`: Number of attempts before moving to the next question.


In [ ]:
# Cell 5: Head Preview & Statistical Summary
print("--- First 5 Records (df.head()): ---")
display(df.head())

print("\n--- Numerical Columns Summary (df.describe()): ---")
display(df.describe().T)


### Cell 6: Missing Values Analysis (`df.isnull().sum()`)


In [ ]:
# Cell 6: Missing Value Profiling
print("--- Missing Values Count per Column: ---")
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({"Missing Rows": missing_counts, "Percentage (%)": missing_pct.round(2)})
display(missing_df)

print(f"\nUnique Students: {df['user_id'].nunique():,}")
print(f"Unique Skills Assessed: {df['skill_name'].dropna().nunique()}")
print("✅ Step 01 Completed Successfully!")
